In [1]:
!uvx mlflow server

⠙ Preparing packages... (0/1)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/32.63 MiB           
⠙ Preparing packages... (0/1)------------------- 16.00 KiB/32.63 MiB         
⠙ Preparing packages... (0/1)------------------- 32.00 KiB/32.63 MiB         
⠹ Preparing packages... (0/1)------------------- 48.00 KiB/32.63 MiB         
⠹ Preparing packages... (0/1)------------------- 48.00 KiB/32.63 MiB         
⠹ Preparing packages... (0/1)------------------- 60.94 KiB/32.63 MiB         
⠹ Preparing packages... (0/1)------------------- 76.94 KiB/32.63 MiB         
⠹ Preparing packages... (0/1)------------------- 92.94 KiB/32.63 MiB         
⠹ Preparing packages... (0/1)------------------- 108.94 KiB/32.63 MiB        
⠹ Preparing packages... (0/1)------------------- 124.94 KiB/32.63 MiB        
⠹ Preparing packages... (0/1)------------------- 140.94 KiB

In [53]:
%pip install mlflow

import mlflow
import mlflow.sklearn
mlflow.set_experiment("f1_podium_prediction")

Note: you may need to restart the kernel to use updated packages.


<Experiment: artifact_location=('file:///Users/liyunxiao/Desktop/QMSS applied data analysis/takehome '
 'exercise/mlruns/329694474544665792'), creation_time=1775956312602, experiment_id='329694474544665792', last_update_time=1775956312602, lifecycle_stage='active', name='f1_podium_prediction', tags={}, trace_location=None, workspace='default'>

In [54]:
pip install pandas numpy matplotlib scikit-learn jupyter

Note: you may need to restart the kernel to use updated packages.


In [55]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [56]:
# read data.   step1
import pandas as pd
import numpy as np
results = pd.read_csv('/Users/liyunxiao/Desktop/QMSS applied data analysis/f1_data/results.csv')
results.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


In [57]:
# check data shape
print(results.shape)
print(results.columns.tolist())
results.info()

(26759, 18)
['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26759 entries, 0 to 26758
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   resultId         26759 non-null  int64  
 1   raceId           26759 non-null  int64  
 2   driverId         26759 non-null  int64  
 3   constructorId    26759 non-null  int64  
 4   number           26759 non-null  object 
 5   grid             26759 non-null  int64  
 6   position         26759 non-null  object 
 7   positionText     26759 non-null  object 
 8   positionOrder    26759 non-null  int64  
 9   points           26759 non-null  float64
 10  laps             26759 non-null  int64  
 11  time             26759 non-null  object 
 12  mi

In [58]:
# create a target variable podium, which is 1 if the driver finished in the top 3 and 0 otherwise. step2
results["podium"] = np.where(results["positionOrder"] <= 3, 1, 0)
results[["positionOrder", "podium"]].head(10)

,positionOrder,podium
0,1,1
1,2,1
2,3,1
3,4,0
4,5,0
5,6,0
6,7,0
7,8,0
8,9,0
9,10,0


In [59]:
print(results["podium"].value_counts())
print(results["podium"].value_counts(normalize=True))

podium
0    23362
1     3397
Name: count, dtype: int64
podium
0    0.873052
1    0.126948
Name: proportion, dtype: float64


In [60]:
model_df = results[["grid", "driverId", "constructorId", "podium"]].copy()
model_df.head()

,grid,driverId,constructorId,podium
0,1,1,1,1
1,5,2,2,1
2,7,3,3,1
3,11,4,4,0
4,3,5,1,0


In [61]:
# check for missing values
model_df.isnull().sum()

grid             0
driverId         0
constructorId    0
podium           0
dtype: int64

In [62]:
# step3: split data into training and testing sets
X = model_df[["grid", "driverId", "constructorId"]]
y = model_df["podium"]

In [63]:
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [64]:
# Check for the data
print(X_train.shape)
print(X_test.shape)
print(y_train.mean())
print(y_test.mean())

(21407, 3)
(5352, 3)
0.12696781426636147
0.12686846038863975


In [65]:
# Step4：creat a baseline model(ramdom forest cllassifier)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [66]:
#Step5: predict and evaluate the model
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

In [67]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.897982062780269
Precision: 0.6625916870415648
Recall: 0.39911634756995584
F1 Score: 0.49816176470588236


In [68]:
print(classification_report(y_test, y_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.92      0.97      0.94      4673
           1       0.66      0.40      0.50       679

    accuracy                           0.90      5352
   macro avg       0.79      0.68      0.72      5352
weighted avg       0.89      0.90      0.89      5352



In [69]:
#Step 6: MLflow
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("f1_podium_prediction")

<Experiment: artifact_location=('file:///Users/liyunxiao/Desktop/QMSS applied data analysis/takehome '
 'exercise/mlruns/329694474544665792'), creation_time=1775956312602, experiment_id='329694474544665792', last_update_time=1775956312602, lifecycle_stage='active', name='f1_podium_prediction', tags={}, trace_location=None, workspace='default'>

In [70]:
from sklearn.metrics import roc_auc_score
with mlflow.start_run(run_name="baseline_rf"):
    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(X_train, y_train)

    y_pred = rf_model.predict(X_test)
    y_prob = rf_model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)


    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("min_samples_split", 5)
    mlflow.log_param("min_samples_leaf", 2)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    auc = roc_auc_score(y_test, y_prob)


    mlflow.sklearn.log_model(rf_model, artifact_path="model")

2026/04/11 22:23:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/11 22:23:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [71]:
# Add artifacts
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix")
plt.savefig("confusion_matrix.png", bbox_inches="tight")
plt.close()

mlflow.log_artifact("confusion_matrix.png")

In [72]:
# feature importance
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.plot(
    x="feature",
    y="importance",
    kind="bar",
    legend=False
)
plt.title("Feature Importance")
plt.ylabel("Importance")
plt.savefig("feature_importance.png", bbox_inches="tight")
plt.close()

mlflow.log_artifact("feature_importance.png")

In [73]:
# Save predictions
predictions_df = X_test.copy()
predictions_df["actual"] = y_test.values
predictions_df["predicted"] = y_pred
predictions_df["predicted_prob"] = y_prob

predictions_df.to_csv("predictions.csv", index=False)
mlflow.log_artifact("predictions.csv")

In [74]:
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [5, 10, 15],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

In [75]:
# Hyperparameter tuning with MLflow.  I pack them into a function so we don't need to write 10 times. 
from sklearn.model_selection import ParameterGrid

def train_and_log_rf(params):
    with mlflow.start_run():
        rf_model = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=42,
            n_jobs=-1
        )

        rf_model.fit(X_train, y_train)

        y_pred = rf_model.predict(X_test)
        y_prob = rf_model.predict_proba(X_test)[:, 1]

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        auc = roc_auc_score(y_test, y_prob)

        mlflow.log_params(params)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", auc)

        mlflow.sklearn.log_model(rf_model, artifact_path="model")

        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot()
        plt.title("Confusion Matrix")
        plt.savefig("confusion_matrix.png", bbox_inches="tight")
        plt.close()
        mlflow.log_artifact("confusion_matrix.png")

        feature_importance = pd.DataFrame({
            "feature": X_train.columns,
            "importance": rf_model.feature_importances_
        }).sort_values("importance", ascending=False)

        feature_importance.plot(
            x="feature",
            y="importance",
            kind="bar",
            legend=False
        )
        plt.title("Feature Importance")
        plt.ylabel("Importance")
        plt.savefig("feature_importance.png", bbox_inches="tight")
        plt.close()
        mlflow.log_artifact("feature_importance.png")

        predictions_df = X_test.copy()
        predictions_df["actual"] = y_test.values
        predictions_df["predicted"] = y_pred
        predictions_df["predicted_prob"] = y_prob
        predictions_df.to_csv("predictions.csv", index=False)
        mlflow.log_artifact("predictions.csv")

        return {
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "min_samples_split": params["min_samples_split"],
            "min_samples_leaf": params["min_samples_leaf"],
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "roc_auc": auc
        }


In [76]:
all_params = list(ParameterGrid(param_grid))
len(all_params)

36

In [77]:
results_list = []

if mlflow.active_run() is not None:
    mlflow.end_run()

for params in all_params[:10]:
    run_result = train_and_log_rf(params)
    results_list.append(run_result)

results_df = pd.DataFrame(results_list)
results_df

2026/04/11 22:23:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/11 22:23:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/11 22:23:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/11 22:23:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

,n_estimators,max_depth,min_samples_split,min_samples_leaf,accuracy,precision,recall,f1_score,roc_auc
0,50,5,2,1,0.889013,0.722513,0.203240,0.317241,0.889216
1,100,5,2,1,0.889387,0.691630,0.231222,0.346578,0.890188
2,150,5,2,1,0.892003,0.701195,0.259205,0.378495,0.890681
3,50,5,5,1,0.889574,0.713592,0.216495,0.332203,0.889003
4,100,5,5,1,0.889761,0.690987,0.237113,0.353070,0.890665
5,150,5,5,1,0.890882,0.687747,0.256259,0.373391,0.891148
6,50,5,2,2,0.889948,0.708333,0.225331,0.341899,0.889791
7,100,5,2,2,0.889761,0.686192,0.241532,0.357298,0.890337
8,150,5,2,2,0.891069,0.701681,0.245950,0.364231,0.890916
9,50,5,5,2,0.889574,0.698198,0.228277,0.344062,0.889747


In [78]:
best_run = results_df.sort_values("f1_score", ascending=False).iloc[0]
best_run

n_estimators         150.000000
max_depth              5.000000
min_samples_split      2.000000
min_samples_leaf       1.000000
accuracy               0.892003
precision              0.701195
recall                 0.259205
f1_score               0.378495
roc_auc                0.890681
Name: 2, dtype: float64

In [79]:
import sys
print(sys.executable)

/Users/liyunxiao/Desktop/QMSS applied data analysis/.venv/bin/python


In [1]:
import mlflow
print("mlflow imported successfully")

mlflow imported successfully


In [2]:
import mlflow

mlflow.set_experiment("test_experiment")

with mlflow.start_run():
    mlflow.log_param("model_type", "demo")
    mlflow.log_metric("accuracy", 0.95)

/Users/liyunxiao/Desktop/QMSS applied data analysis/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/11 22:29:21 INFO mlflow.tracking.fluent: Experiment with name 'test_experiment' does not exist. Creating a new experiment.
